# You are building a classification model to predict flower type using the Iris dataset.

Input → flower features (length, width)

Output → flower type (Setosa, Versicolor, Virginica)

# Loading Dataset

In [21]:

import torch
import torch.nn as nn
import torch.optim as optim

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import seaborn as sns


In [1]:
from sklearn.datasets import load_iris

data = load_iris()
X = data.data
y = data.target

In [2]:
X

array([[5.1, 3.5, 1.4, 0.2],
       [4.9, 3. , 1.4, 0.2],
       [4.7, 3.2, 1.3, 0.2],
       [4.6, 3.1, 1.5, 0.2],
       [5. , 3.6, 1.4, 0.2],
       [5.4, 3.9, 1.7, 0.4],
       [4.6, 3.4, 1.4, 0.3],
       [5. , 3.4, 1.5, 0.2],
       [4.4, 2.9, 1.4, 0.2],
       [4.9, 3.1, 1.5, 0.1],
       [5.4, 3.7, 1.5, 0.2],
       [4.8, 3.4, 1.6, 0.2],
       [4.8, 3. , 1.4, 0.1],
       [4.3, 3. , 1.1, 0.1],
       [5.8, 4. , 1.2, 0.2],
       [5.7, 4.4, 1.5, 0.4],
       [5.4, 3.9, 1.3, 0.4],
       [5.1, 3.5, 1.4, 0.3],
       [5.7, 3.8, 1.7, 0.3],
       [5.1, 3.8, 1.5, 0.3],
       [5.4, 3.4, 1.7, 0.2],
       [5.1, 3.7, 1.5, 0.4],
       [4.6, 3.6, 1. , 0.2],
       [5.1, 3.3, 1.7, 0.5],
       [4.8, 3.4, 1.9, 0.2],
       [5. , 3. , 1.6, 0.2],
       [5. , 3.4, 1.6, 0.4],
       [5.2, 3.5, 1.5, 0.2],
       [5.2, 3.4, 1.4, 0.2],
       [4.7, 3.2, 1.6, 0.2],
       [4.8, 3.1, 1.6, 0.2],
       [5.4, 3.4, 1.5, 0.4],
       [5.2, 4.1, 1.5, 0.1],
       [5.5, 4.2, 1.4, 0.2],
       [4.9, 3

In [4]:
data.target_names

array(['setosa', 'versicolor', 'virginica'], dtype='<U10')

In [6]:
data.target # numpy array (150,)  values: 0, 1, 2

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2])

In [8]:
print(f"X shape : {X.shape}  →  {X.shape[0]} samples, {X.shape[1]} features")
print(f"Classes : {list(data.target_names)}")
print()

X shape : (150, 4)  →  150 samples, 4 features
Classes : [np.str_('setosa'), np.str_('versicolor'), np.str_('virginica')]



# Train Test split

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

The function train_test_split is used to divide the dataset into two parts: training data and testing data. Here, X represents the input features (such as age, BMI, etc.), and y represents the target variable (such as stroke or class label). The parameter test_size=0.2 means that 20% of the data is used for testing and 80% is used for training. The random_state=42 ensures that the split is the same every time the code is run, making the results reproducible. The stratify=y parameter ensures that the proportion of each class in the target variable remains the same in both the training and testing datasets.

After splitting, X_train contains the input features used to train the model, and y_train contains the corresponding correct outputs for that training data. Similarly, X_test contains the input features used to evaluate the model, and y_test contains the actual correct outputs for the test data. During training, the model learns patterns using X_train and y_train, and during testing, it makes predictions on X_test, which are then compared with y_test to measure how well the model performs on unseen data.

In [10]:
print(f"Train : {X_train.shape}  ({len(X_train)} samples)")
print(f"Test  : {X_test.shape}   ({len(X_test)} samples)")
print()

Train : (120, 4)  (120 samples)
Test  : (30, 4)   (30 samples)



In [13]:
#Scale using training statistics ONLY ──────────────────────────────────
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)  # Learns μ and σ from training data
X_test  = scaler.transform(X_test)       # Applies training μ/σ to test — no leakage!

print(f"After scaling — Train mean : {X_train.mean(axis=0).round(2)}  (should be ≈ 0)")
print(f"After scaling — Train std  : {X_train.std(axis=0).round(2)}   (should be ≈ 1)")
print()

After scaling — Train mean : [-0. -0.  0.  0.]  (should be ≈ 0)
After scaling — Train std  : [1. 1. 1. 1.]   (should be ≈ 1)



In [14]:
from sklearn.preprocessing import StandardScaler

In [19]:
# ── 3c: Convert to tensors ────────────────────────────────────────────────────
import torch
X_train_t = torch.tensor(X_train, dtype=torch.float32)  # float32 for features
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)     # int64 for class labels!
y_test_t  = torch.tensor(y_test,  dtype=torch.long)

print(f"X_train_t : {X_train_t.shape}  dtype={X_train_t.dtype}")
print(f"y_train_t : {y_train_t.shape}   dtype={y_train_t.dtype}  ← must be long for CrossEntropyLoss")

X_train_t : torch.Size([120, 4])  dtype=torch.float32
y_train_t : torch.Size([120])   dtype=torch.int64  ← must be long for CrossEntropyLoss


/tmp/ipykernel_16543/2237940024.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_train_t = torch.tensor(X_train, dtype=torch.float32)  # float32 for features
/tmp/ipykernel_16543/2237940024.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
/tmp/ipykernel_16543/2237940024.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_train_t = torch.tensor(y_train, dtype=torch.long)     # int64 for class labels!
/tmp/ipykernel_16543/2237940024.py:6: UserWarning: To copy construct from a 

In [25]:
# Model Definition
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(4, 3)

    def forward(self, x):
        return self.fc(x)

        model = Model()   # ✅ create model instance

This code defines a simple neural network model using PyTorch by creating a class that inherits from nn.Module. The __init__ method is the constructor where the model’s layers are defined, and super().__init__() ensures that the base class is properly initialized so PyTorch can track the model’s parameters. Inside it, self.fc = nn.Linear(4, 3) creates a fully connected (linear) layer that takes 4 input features and produces 3 output values, which typically represent the scores (logits) for 3 different classes in a classification problem. The forward method defines how the input data flows through the model; when you pass data x to the model, it simply applies the linear layer (self.fc(x)) and returns the result. In simple terms, this model learns a linear relationship that maps 4 input features to 3 output classes.

4 inputs -> 3 outputs => 3 flower classes

In [23]:
# Loss Function

loss_fn = nn.CrossEntropyLoss()

The line loss_fn = nn.CrossEntropyLoss() defines the loss function used for a classification problem in PyTorch. A loss function measures how wrong the model’s predictions are compared to the actual correct labels. In this case, CrossEntropyLoss is specifically designed for multi-class classification tasks, where the model predicts one class out of several possible classes (like the Iris dataset with 3 flower types). The model outputs raw scores (called logits) for each class, and CrossEntropyLoss internally applies a softmax function to convert these scores into probabilities and then compares them with the true class labels. It assigns a higher loss when the predicted class is far from the actual class and a lower loss when the prediction is correct. During training, the goal is to minimize this loss, which helps the model improve its predictions over time.

In [27]:
# Optimiser
model = Model()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

This code creates a model and defines an optimizer to train it. The line model = Model() creates an instance of the neural network you defined earlier, which contains the learnable parameters (weights and biases). These parameters are what the model will adjust during training to learn patterns from the data. The next line, optimizer = torch.optim.SGD(model.parameters(), lr=0.01), defines the optimizer using Stochastic Gradient Descent (SGD). The optimizer takes the model’s parameters (model.parameters()) and updates them during training based on the computed gradients. The lr=0.01 is the learning rate, which controls how big the update step should be when adjusting the weights. In simple terms, the model makes predictions, the optimizer helps improve the model by updating its weights step-by-step to reduce the error.

In [29]:
# Training loop

epochs = 100

for epoch in range(epochs):

    # 1. forward
    y_pred = model(X_train)

    # 2. loss
    loss = loss_fn(y_pred, y_train)

    # 3. zero grad
    optimizer.zero_grad()

    # 4. backward
    loss.backward()

    # 5. update
    optimizer.step()

In [30]:
# Prediction

with torch.no_grad():
    predictions = model(X_test)

    # no_grad() = no training, only testing

In [31]:
# Convert Output to Classes - This picks the class with highest probability
_, predicted = torch.max(predictions, 1)


In [32]:
# Accuracy prediction - Checks how many predictions are correct

accuracy = (predicted == y_test).sum() / len(y_test)

In [33]:
accuracy

tensor(0.7667)

# Data → Split → Tensor → Model → Train → Predict → Accuracy